# Generación del Knowledge Base sintético (Fase 2.1)

**Proyecto Final — Asistente de Triaje de Mails con RAG y LLM**

---

## Objetivo

Generar 30 FAQs realistas para una empresa ficticia (**ShopHive**), cubriendo las 5 categorías × 6 temas, usando Gemini como redactor.

Las FAQs se guardan como `.md` con frontmatter YAML (categoría, tema, idioma, modelo que las generó) para trazabilidad — el informe puede mostrar exactamente cómo se construyó el KB.

## Por qué generamos sintético en vez de usar un dataset público

1. **Control total sobre las FAQs:** sabemos exactamente qué cubre cada documento.
2. **Trazabilidad para evaluación:** podemos construir un ground truth manual cruzando consultas de prueba con las FAQs que las responden.
3. **Sin sesgos de dominio externo:** no arrastramos vocabulario de otra empresa.
4. **Demuestra una capacidad real:** generar datos sintéticos con LLM para arrancar un sistema RAG es un patrón muy usado en producción.

## 1. Setup (Colab + Local)

In [ ]:
%%capture
!pip install -q -U google-genai pydantic python-dotenv pandas
!pip uninstall -y google-generativeai 2>/dev/null || true

In [ ]:
# Bootstrap: idéntico al del notebook anterior
REPO_URL = "https://github.com/<TU-USUARIO>/<TU-REPO>.git"  # ⚠️ cambialo por tu URL

import os, sys, json, time
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    repo_name = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
    if not Path(repo_name).exists():
        os.system(f'git clone {REPO_URL}')
    os.chdir(repo_name)
    PROJECT_ROOT = Path.cwd()
    try:
        from google.colab import userdata
        os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    except Exception:
        import getpass
        os.environ['GOOGLE_API_KEY'] = getpass.getpass('Pegá tu GOOGLE_API_KEY: ')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    from dotenv import load_dotenv
    load_dotenv(PROJECT_ROOT / '.env')

sys.path.insert(0, str(PROJECT_ROOT))

assert os.getenv('GOOGLE_API_KEY'), 'Falta GOOGLE_API_KEY'
print(f'✅ Entorno: {"Colab" if IN_COLAB else "Local"}')
print(f'✅ Project root: {PROJECT_ROOT}')

## 2. Generar todas las FAQs

Esto hace **~30 llamadas a Gemini**, pausando 5 segundos entre cada una para respetar el rate limit. Total aprox: **3-5 minutos**.

Si la celda se interrumpe (rate limit, network), basta con volver a correrla: `skip_existing=True` evita regenerar archivos que ya existen.

In [ ]:
from src.kb_generate import generate_all_faqs, TOPICS

print(f'Voy a generar {len(TOPICS)} FAQs distribuidas así:')
from collections import Counter
for cat, count in Counter(t[0] for t in TOPICS).items():
    print(f'  - {cat}: {count} temas')

In [ ]:
generated = generate_all_faqs(
    output_dir='data/kb',
    sleep_between_calls_s=5.0,
    skip_existing=True,
    verbose=True,
)
print(f'\n📦 Total final: {len(generated)} FAQs en data/kb/')

## 3. Preview de las FAQs generadas

Echamos un vistazo a 3 FAQs al azar para validar calidad.

In [ ]:
import random
from pathlib import Path

kb_dir = PROJECT_ROOT / 'data/kb'
files = sorted(kb_dir.glob('*.md'))
sample = random.sample(files, min(3, len(files)))

for f in sample:
    print('=' * 70)
    print(f.name)
    print('=' * 70)
    print(f.read_text(encoding='utf-8'))
    print()

## 4. Distribución del KB

Verificamos que las 5 categorías estén bien representadas.

In [ ]:
import pandas as pd
from src.kb_ingest import _parse_md

rows = []
for f in files:
    meta = _parse_md(f)
    rows.append({
        'archivo':  f.name,
        'categoría': meta.get('category', 'UNKNOWN'),
        'tema':     meta.get('topic', ''),
        'palabras': len(meta.get('body', '').split()),
    })
df = pd.DataFrame(rows)

print(f'Total FAQs:      {len(df)}')
print(f'Palabras medias: {df["palabras"].mean():.0f}')
print(f'Palabras min/max: {df["palabras"].min()} / {df["palabras"].max()}')
print()
print('Por categoría:')
print(df.groupby('categoría').size().to_string())

## 5. (Solo Colab) — Descargar el KB para commitear al repo

Esto zipea la carpeta `data/kb/` y la baja a tu máquina. Después la descomprimís en tu repo local y haces `git add data/kb/ && git commit && git push`.

In [ ]:
if IN_COLAB:
    import shutil
    zip_path = '/content/kb_generated.zip'
    shutil.make_archive('/content/kb_generated', 'zip', root_dir=str(PROJECT_ROOT / 'data'), base_dir='kb')
    from google.colab import files
    files.download(zip_path)
else:
    print('Estás corriendo local — los archivos ya están en data/kb/. Hacé git add y commit.')

## 6. Próximo paso

Con el KB ya en el repo, pasamos al notebook `04_rag_demo.ipynb`, donde:

1. Indexamos las 30 FAQs en Chroma (vector) + BM25 (léxico).
2. Implementamos búsqueda híbrida con Reciprocal Rank Fusion.
3. Probamos consultas en EN y ES contra el KB.
4. Vemos qué documentos recupera para cada consulta — base para medir Recall@k y Precision@k en la Fase 4.